# 04 · Current retrospective benchmark and optional export

This notebook evaluates the **exact completed feature and model runs from 02 and 03**. It refits the predeclared men's ranking-logistic and women's logistic anchors for 2022–2025, with all choices frozen from pre-2022 development predictions. These years have already been consumed: this is a retrospective comparison, not an untouched holdout.

`MODE = "review"` verifies and displays committed evidence. `MODE = "train"` runs the 16 annual benchmark fits with verified checkpoints. Neither mode generates a 2026 submission; the final export cell remains a separate, default-off option.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from march_mania.notebook_support import table, style
from march_mania.runtime import digest
from march_mania.publication.workflow import benchmark_evidence, benchmark_stage, execution_mode

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
MODE = execution_mode()  # Set "train" to evaluate the current completed 02/03 runs.
if MODE == "train":
    benchmark_stage(ROOT)
FINAL, record = benchmark_evidence(ROOT)
summary = record["summary"]
assert summary["status"] == "completed" and not summary["future_labels_used"]
assert not summary["submission_generated"]
style()
display(Markdown(f"**{summary['model_fits']} annual fits · {summary['physical_games']} physical games · retrospective 2022–2025 benchmark**"))
table(pd.DataFrame([{"Stage": stage, "Fingerprint": record["manifest"]["inputs"][key]}
                    for stage, key in [("02 features", "feature_fingerprint"), ("03 models", "model_fingerprint")]]))


## Frozen recipe and training boundaries

The model families and identity calibration follow the predeclared inference contract; notebook 03 separately compares all supported families. Within each logistic family, the feature block and regularization use only saved 2016–2021 development predictions. Every evaluation season is fitted on earlier seasons only. The seed-free route removes every seed-dependent input and uses the same regularization.


In [ ]:
recipe = json.loads((FINAL / "recipe.json").read_text())
fits = json.loads((FINAL / "fit_audits.json").read_text())
assert all(r["training_max_season"] < r["prediction_season"] < 2026 for r in fits)
table(pd.DataFrame([{"Tournament": r["gender"], "Route": r["route"],
                     "Evaluated season": r["prediction_season"],
                     "Candidate": r["candidate"]["name"],
                     "Candidates": r["candidate_count"], "Retained": r["retained_count"],
                     "Training games": r["training_games"],
                     "Last training season": r["training_max_season"]} for r in fits]))


## Retrospective 2022–2025 benchmark

Each season is predicted using only earlier seasons, with the same frozen recipe. The seed-free route is also evaluated on the same tournament games to expose the value of seed information. Its tournament performance does not prove generalization to teams that never qualified.

**Game-weighted Brier** averages every game's squared error. **Mean season Brier** weights seasons equally. Log loss, ROC AUC, average precision and calibration error provide complementary diagnostics. The benchmark was consumed in prior work and is not used to replace the frozen recipe.

The expanded seeded anchors score **0.198288 for men and 0.146881 for women**. The previous 124-feature anchors scored 0.198014 and 0.138599. Neither improves here, and the women regress materially. The new conference block’s development minimum did not establish better benchmark performance. This negative result remains visible without reselecting on the consumed benchmark.

In [ ]:
current = pd.read_csv(FINAL / "metrics.csv")
seasonal = pd.read_csv(FINAL / "metrics_by_season.csv")
table(current[["Gender", "route", "games", "brier", "mean_season_brier",
               "log_loss", "roc_auc", "average_precision", "ece_10_bins"]])
fig, ax = plt.subplots(figsize=(9, 4.6), constrained_layout=True)
for (gender, route), group in seasonal.groupby(["Gender", "route"]):
    group = group.sort_values("Season")
    ax.plot(group.Season, group.brier, marker="o",
            linestyle="-" if route == "seeded" else "--",
            label=f"{'Men' if gender == 'M' else 'Women'} · {route.replace('_',' ')}")
ax.set(title="Frozen recipe across previously consumed seasons",
       xlabel="Predicted tournament season", ylabel="Brier score · lower is better",
       xticks=[2022, 2023, 2024, 2025])
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=.2)
ax.legend(frameon=False, ncol=2, fontsize=9)
plt.show()

In [ ]:
reliability = pd.read_csv(FINAL / "reliability.csv")
fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
ax.plot([0,1], [0,1], linestyle="--", alpha=.5, label="Perfect calibration")
for gender, group in reliability.loc[reliability.route.eq("seeded")].groupby("Gender"):
    ax.plot(group.predicted, group.observed, marker="o",
            label=f"{'Men' if gender == 'M' else 'Women'} · seeded")
ax.set(title="Calibration · retrospective seeded tournament forecasts",
       xlabel="Mean predicted win probability", ylabel="Observed win frequency",
       xlim=(0,1), ylim=(0,1))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=.15)
ax.legend(frameon=False)
plt.show()
display(Markdown("Points summarize ten probability bins. Small bin counts and only four "
                 "seasons limit precision; proximity to the diagonal is not proof of "
                 "future calibration. No curve here is used for post-benchmark tuning."))

## Evidence and reproducibility

Each of the 16 estimators and 16 prediction batches is checkpointed after its output hashes are recorded. The portable archive preserves the complete audit, development predictions, fitted models, source and feature matrix. The independent model-search recovery test in notebook 03 reports whether a fresh directory reproduces forecasts without repeating fits.

Historical final-submission evidence remains under `reports/final_predictions/` with its original 124-feature lineage. It is not evidence for this expanded experiment.


In [ ]:
archive = record["archive"]
assert archive["status"] == "uploaded"
table(pd.DataFrame([{"Run": summary["fingerprint"], "Archive SHA-256": archive["sha256"],
                     "Bytes": archive["bytes"], "Version": archive["version_id"],
                     "Submission generated": summary["submission_generated"],
                     "Kaggle upload sent": summary["kaggle_submission_sent"]}]))
display(Markdown("The archive and exact upstream fingerprints make this evaluation reproducible. The benchmark is descriptive; it does not authorize tuning on these seasons."))


## Generate and download your own file

First run **02 and 03 with `MODE = "train"`**, using your current raw inputs and matching checkpoints. Then set `GENERATE_SUBMISSION = True` below and run the cell. It calls real model refitting/inference when needed, verifies exact template order, probability bounds and team orientation, then writes **`submissions/submission.csv`** atomically. Matching successful fits are reused. It requires the existing AWS role to retain durable checkpoints.

The cell refuses stale feature or model inputs rather than exporting probabilities from an unrelated saved experiment. The recipe is explicit in `configs/inference.json`; the existing logistic recipe is a frozen baseline, not a claim that it wins every later research comparison.

Use the displayed link, or find `submissions/submission.csv` in JupyterLab's file browser and choose **Download**. You decide whether to upload it to Kaggle. There is no upload call in this workflow. This notebook does not create a CSV in review mode.

In [ ]:
from IPython.display import HTML
from march_mania.publication.workflow import generate_submission

GENERATE_SUBMISSION = False  # Change to True only when YOU are ready to generate locally.
if GENERATE_SUBMISSION:
    SUBMISSION = generate_submission(ROOT)
    display(Markdown(f"**Your file is ready:** `submissions/submission.csv`  \nSHA-256: `{digest(SUBMISSION)}`. No Kaggle upload was sent."))
    display(HTML('<a href="../submissions/submission.csv" download="submission.csv">Download your submission.csv</a>'))
else:
    display(Markdown("**Export is off.** Set `GENERATE_SUBMISSION = True` to generate and download your own file after 02/03 training."))

Sources: [official competition and evaluation](https://www.kaggle.com/competitions/march-machine-learning-mania-2026) · exact current benchmark manifests in `reports/benchmark/` · historical [score log](../reports/submission_portfolio/). Local evaluation includes play-ins and is not identical to Kaggle's scored population.
